In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train_data.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
test_data.head()


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
# -*- coding: utf-8 -*-
"""
Titanic: Advanced Solution with Stacking and Fine-Tuning
Expected Public LB Score: 0.82 - 0.84
"""

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# -------------------------------
# 1. Загрузка и первичный осмотр
# -------------------------------
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
PassengerId = test['PassengerId']

# Объединяем для единой обработки
all_data = pd.concat([train, test], sort=False).reset_index(drop=True)

# -------------------------------
# 2. Расширенный Feature Engineering
# -------------------------------

# 2.1 Титулы (расширенная версия)
all_data['Title'] = all_data['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())
title_map = {
    "Mr": "Mr", "Miss": "Miss", "Mrs": "Mrs", "Master": "Master",
    "Dr": "Officer", "Rev": "Officer", "Col": "Officer", "Major": "Officer",
    "Mlle": "Miss", "Ms": "Miss", "Lady": "Royalty", "Sir": "Royalty",
    "Mme": "Mrs", "Capt": "Officer", "Don": "Royalty", "Jonkheer": "Royalty",
    "Countess": "Royalty", "Dona": "Royalty"
}
all_data['Title'] = all_data['Title'].map(title_map)

# 2.2 Фамилия (для группировки семей)
all_data['Surname'] = all_data['Name'].apply(lambda x: x.split(',')[0].strip())

# 2.3 Признак «спасся в шлюпке» (из Name, где указан номер шлюпки)
all_data['Boat'] = all_data['Name'].str.extract(r'\(([^)]+)\)')[0]
# Если в скобках есть число — возможно, номер шлюпки
all_data['HasBoatInfo'] = all_data['Boat'].notna().astype(int)
all_data.drop('Boat', axis=1, inplace=True)

# 2.4 Обработка кают (заполнение по соседям с таким же билетом)
# Сначала выделим палубу (первая буква)
all_data['Deck'] = all_data['Cabin'].str[0]
# Заполним пропуски в Deck на основе билета (если у пассажиров одинаковый билет, то Deck одинаков)
deck_by_ticket = all_data.groupby('Ticket')['Deck'].apply(lambda x: x.mode()[0] if not x.mode().empty else 'U')
all_data['Deck'] = all_data.apply(lambda row: deck_by_ticket[row['Ticket']] if pd.isna(row['Deck']) else row['Deck'], axis=1)
all_data['Deck'] = all_data['Deck'].fillna('U')
# Бинарный признак: есть ли каюта вообще
all_data['HasCabin'] = (all_data['Deck'] != 'U').astype(int)

# 2.5 Билет: размер группы по одинаковому билету
ticket_group = all_data.groupby('Ticket')['PassengerId'].count().reset_index()
ticket_group.columns = ['Ticket', 'TicketGroupSize']
all_data = all_data.merge(ticket_group, on='Ticket', how='left')

# 2.6 Семья: объединяем SibSp, Parch и TicketGroup
all_data['FamilySize'] = all_data['SibSp'] + all_data['Parch'] + 1
all_data['IsAlone'] = (all_data['FamilySize'] == 1).astype(int)
all_data['LargeFamily'] = (all_data['FamilySize'] >= 5).astype(int)

# 2.7 Возраст: заполняем медианой по Title
all_data['Age'] = all_data.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
# Категориальный возраст
all_data['AgeBin'] = pd.cut(all_data['Age'], bins=[0, 12, 18, 30, 50, 65, 100], labels=False)
all_data['IsChild'] = (all_data['Age'] < 12).astype(int)
all_data['IsElderly'] = (all_data['Age'] > 60).astype(int)

# 2.8 Fare: заполняем медианой по Pclass
all_data['Fare'] = all_data.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
# Логарифмирование тарифа (убирает выбросы)
all_data['LogFare'] = np.log1p(all_data['Fare'])

# 2.9 Embarked: заполняем модой
all_data['Embarked'] = all_data['Embarked'].fillna(all_data['Embarked'].mode()[0])

# 2.10 Дополнительные взаимодействия
all_data['Sex_Pclass'] = all_data['Sex'] + '_' + all_data['Pclass'].astype(str)
all_data['Title_Pclass'] = all_data['Title'] + '_' + all_data['Pclass'].astype(str)

# Удаляем ненужные колонки
drop_cols = ['Name', 'Cabin', 'Ticket', 'Surname', 'PassengerId']
all_data.drop([c for c in drop_cols if c in all_data.columns], axis=1, inplace=True)

# -------------------------------
# 3. Кодирование категорий
# -------------------------------
# Label Encoding для высококардинальных
le = LabelEncoder()
for col in ['Title', 'Deck', 'Sex_Pclass', 'Title_Pclass']:
    all_data[col] = le.fit_transform(all_data[col])

# One-Hot для бинарных/низкокардинальных
all_data = pd.get_dummies(all_data, columns=['Sex', 'Embarked'], drop_first=True)

# -------------------------------
# 4. Разделение обратно
# -------------------------------
train_processed = all_data[all_data['Survived'].notna()].copy()
test_processed = all_data[all_data['Survived'].isna()].copy()
test_processed.drop('Survived', axis=1, inplace=True)

X = train_processed.drop('Survived', axis=1)
y = train_processed['Survived'].astype(int)

# Проверка и заполнение оставшихся NaN (на всякий случай)
X.fillna(X.median(numeric_only=True), inplace=True)
X.fillna(X.mode().iloc[0], inplace=True)
test_processed.fillna(test_processed.median(numeric_only=True), inplace=True)
test_processed.fillna(test_processed.mode().iloc[0], inplace=True)

# Масштабирование
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test_processed)

# -------------------------------
# 5. Подбор гиперпараметров (GridSearch) – выполняется один раз
# -------------------------------
# Закомментируйте этот блок после первого запуска, чтобы экономить время.
# Лучшие параметры уже подобраны и вставлены в модели ниже.

"""
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4]
}
rf = RandomForestClassifier(random_state=42)
grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X, y)
print("Best RF params:", grid_rf.best_params_)
# Best RF params: {'max_depth': 6, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
"""

# Аналогично для XGBoost и LightGBM (опущено для краткости)

# -------------------------------
# 6. Базовые модели с подобранными параметрами
# -------------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_split=5,
        min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        use_label_encoder=False, eval_metric='logloss'
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        random_state=42
    ),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(probability=True, kernel='rbf', C=1.0, gamma='scale', random_state=42)
}

# Оценка каждой модели кросс-валидацией
print("=== Cross-Validation Accuracy ===")
for name, model in models.items():
    if name in ['LogisticRegression', 'KNN', 'SVM']:
        scores = cross_val_score(model, X_scaled, y, cv=kfold, scoring='accuracy')
    else:
        scores = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')
    print(f"{name:20s}: {scores.mean():.4f} (+/- {scores.std():.4f})")

# -------------------------------
# 7. Стекинг (Stacking) с Out-of-Fold предсказаниями
# -------------------------------
def get_oof_predictions(model, X, y, test_X, scaled=False):
    """Генерирует out-of-fold предсказания для train и усреднённые для test."""
    oof_train = np.zeros(len(X))
    oof_test = np.zeros(len(test_X))
    oof_test_skf = np.zeros((len(test_X), kfold.n_splits))

    for i, (train_idx, val_idx) in enumerate(kfold.split(X, y)):
        X_tr = X.iloc[train_idx] if not scaled else X[train_idx]
        X_val = X.iloc[val_idx] if not scaled else X[val_idx]
        y_tr = y.iloc[train_idx]

        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_tr, y_tr)
        oof_train[val_idx] = model_clone.predict_proba(X_val)[:, 1]
        oof_test_skf[:, i] = model_clone.predict_proba(test_X)[:, 1]

    oof_test = oof_test_skf.mean(axis=1)
    return oof_train, oof_test

# Собираем OOF-предсказания от каждой модели
meta_features_train = np.zeros((X.shape[0], len(models)))
meta_features_test = np.zeros((test_processed.shape[0], len(models)))

for idx, (name, model) in enumerate(models.items()):
    use_scaled = name in ['LogisticRegression', 'KNN', 'SVM']
    X_input = X_scaled if use_scaled else X
    test_input = test_scaled if use_scaled else test_processed
    oof_train, oof_test = get_oof_predictions(model, X_input, y, test_input, scaled=use_scaled)
    meta_features_train[:, idx] = oof_train
    meta_features_test[:, idx] = oof_test

# Мета-модель (Logistic Regression на OOF-предсказаниях)
meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_model.fit(meta_features_train, y)

# Финальные предсказания
final_proba = meta_model.predict_proba(meta_features_test)[:, 1]
final_pred = (final_proba >= 0.5).astype(int)

# Оценка стекинга кросс-валидацией (на OOF)
stack_cv_score = cross_val_score(meta_model, meta_features_train, y, cv=kfold, scoring='accuracy')
print(f"\nStacking CV Accuracy: {stack_cv_score.mean():.4f} (+/- {stack_cv_score.std():.4f})")

# -------------------------------
# 8. Сохранение результата
# -------------------------------
submission = pd.DataFrame({'PassengerId': PassengerId, 'Survived': final_pred})
submission.to_csv('submission.csv', index=False)
print("\nФайл submission.csv готов!")

=== Cross-Validation Accuracy ===
RandomForest        : 0.8406 (+/- 0.0060)
XGBoost             : 0.8384 (+/- 0.0155)
LightGBM            : 0.8384 (+/- 0.0188)
GradientBoosting    : 0.8428 (+/- 0.0157)
LogisticRegression  : 0.8215 (+/- 0.0188)
KNN                 : 0.8260 (+/- 0.0140)
SVM                 : 0.8271 (+/- 0.0105)

Stacking CV Accuracy: 0.8384 (+/- 0.0141)

Файл submission.csv готов!
